# Task 3 trên Kaggle T4 ×2 — v3, sửa chẩn đoán và kiểm soát level

Model Qwen/Qwen3-4B-Instruct-2507, NF4/FP16, mỗi GPU một worker. Đây là inference + RAG từ train, không phải fine-tune. Bật Internet và chọn GPU T4 ×2. Add Input source ZIP MỚI, sample_dataset (có exams.json và code), public test và requested levels.

Bản sửa dùng diagnosis_v003 → đối chiếu lại với toàn bộ đề/code → feedback v007. Loại cáo buộc không có bằng chứng hoặc dùng đầu vào ngoài ràng buộc. Level 1 chỉ nhận xét chung, taxonomy vẫn lấy từ chẩn đoán; Level 3 phải có hướng sửa bằng lời. Kiểm tra từ khóa và bước model tự kiểm tra không thay thế đánh giá của người; chưa có điểm leaderboard mới.

Chạy kiểm tra 8 mẫu train có nhãn trước public. RAG loại chính mẫu, code trùng/gần trùng và toàn bộ cùng đề trong lượt kiểm tra này; nhãn/feedback tham chiếu không vào query. Báo cáo label F1 chỉ là tín hiệu phụ, không phải điểm chẩn đoán feedback. Đọc phản hồi và so với giảng viên trước lượt đầy đủ.

Không dùng script phục hồi bên ngoài, không trộn checkpoint cũ. Output mới có hậu tố review1. Tắt fallback chung chung: khi lỗi dai dẳng, giữ partial checkpoint, xem responses.jsonl và sửa nguyên nhân rồi chạy lại. Attention chia query khối 128, không cắt code. Lượt kiểm tra lại làm tăng thời gian inference; giữ cache riêng tư sau phiên Kaggle.


In [ ]:
from pathlib import Path
import sys, subprocess, zipfile, json, os, shutil
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
REPO = WORK / 'LLM-APA-task3'

# Kaggle có thể tự giải nén ZIP khi upload.
sources = sorted({p.parent.parent for p in INPUT.rglob('scripts/predict.py')
                  if (p.parent.parent / 'src/llm_grading').is_dir()
                  and (p.parent.parent / 'tasks/task3_feedback/prompts/v004.txt').is_file()})
archives = list(INPUT.rglob('task3_kaggle_source.zip'))
if len(sources) == 1:
    shutil.copytree(sources[0], REPO, dirs_exist_ok=True)
    print('Đã copy mã nguồn từ:', sources[0])
elif not sources and len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(REPO)
    print('Đã giải nén:', archives[0])
else:
    raise ValueError(f'Cần một bộ mã nguồn duy nhất. Thư mục: {sources}; ZIP: {archives}')

required = ['configs/task3/kaggle_v3.yaml',
    'tasks/task3_feedback/prompts/v007.txt',
    'tasks/task3_feedback/prompts/diagnosis_v003.txt',
    'tasks/task3_feedback/prompts/diagnosis_review_v001.txt']
assert all((REPO/p).is_file() for p in required), 'Upload task3_kaggle_source.zip mới (review1)'
assert 'level1_summary' in (REPO/'src/llm_grading/prompting/pipeline.py').read_text(), 'Source pipeline quá cũ'
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'pyyaml', 'scikit-learn', 'tqdm', 'transformers>=4.56,<5', 'accelerate>=1.0,<2', 'bitsandbytes>=0.46,<1'], check=True)
sys.path[:0] = [str(REPO), str(REPO / 'src')]

def run(script, *args):
    subprocess.run([sys.executable, str(REPO/'scripts'/script), *map(str,args)], check=True)


Nếu cell cài thư viện yêu cầu restart kernel, hãy restart rồi chạy lại các cell trước khi tiếp tục. Giữ PyTorch CUDA có sẵn của Kaggle, không chạy uv sync --all-groups ở notebook này.


In [ ]:
import torch
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'Chưa bật GPU'
assert torch.cuda.device_count() == 2, 'Chọn GPU T4 x2 trong Settings'
for i in range(2):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory / 2**30, 1), 'GiB')


In [ ]:
# Nếu tìm thấy nhiều dataset, điền đường dẫn chính xác ở đây.
TRAIN_JSON = None  # '/kaggle/input/.../sample_dataset/task3_feedback.json'
PUBLIC_JSON = None # '/kaggle/input/.../public_test_release/task3_feedback.json'
LEVELS_JSON = None # '/kaggle/input/.../task3_requested_levels....json'

public_archives = list(INPUT.rglob('public_test.zip'))
if PUBLIC_JSON is None and public_archives:
    assert len(public_archives) == 1
    with zipfile.ZipFile(public_archives[0]) as z:
        z.extractall(WORK/'public-data')
files = list(INPUT.rglob('task3_feedback.json')) + list((WORK/'public-data').rglob('task3_feedback.json'))
def choose(training):
    found=[]
    for p in files:
        rows=json.loads(p.read_text(encoding='utf-8-sig'))['samples']
        if bool(rows and 'output' in rows[0]) == training:
            found.append(p)
    assert len(found)==1, f'Điền đường dẫn thủ công, ứng viên: {found}'
    return found[0]
TRAIN_JSON = Path(TRAIN_JSON) if TRAIN_JSON else choose(True)
PUBLIC_JSON = Path(PUBLIC_JSON) if PUBLIC_JSON else choose(False)
if LEVELS_JSON is None:
    matches=list(INPUT.rglob('task3_requested_levels*.json'))
    assert len(matches)==1, 'Cần file requested levels duy nhất'
    LEVELS_JSON=matches[0]
rows=json.loads(Path(LEVELS_JSON).read_text(encoding='utf-8-sig'))
levels={r['sample_id']:r['requested_level'] for r in rows}
assert len(levels)==len(rows) and all(type(v) is int and v in (1,2,3,4) for v in levels.values())
public=json.loads(PUBLIC_JSON.read_text(encoding='utf-8-sig'))['samples']
assert set(levels)=={r['sample_id'] for r in public}
for r in public:
    assert int(r['input']['target_feedback_level'].split()[1]) == levels[r['sample_id']], r['sample_id']
from llm_grading.data.loader import load_samples
train=load_samples(TRAIN_JSON, task='task3')
assert len(train)==32, f'Train phải có 32 mẫu, thấy {len(train)}'
print('Train:', len(train), 'Public:', len(public), 'Level khớp toàn bộ')


In [ ]:
# Kiểm tra attention với context dài trước khi nạp model.
import subprocess
stress_code = r"""
import sys
sys.path[:0] = [str(REPO_PATH), str(REPO_PATH / 'src')]
import torch
from torch.nn.attention import SDPBackend, sdpa_kernel
from llm_grading.models.generation import _equal_head_sdpa
q=torch.randn(1,32,8192,128,device='cuda',dtype=torch.float16)
k=torch.randn(1,8,8192,128,device='cuda',dtype=torch.float16)
v=torch.randn_like(k)
torch.cuda.reset_peak_memory_stats()
with torch.inference_mode(), _equal_head_sdpa(128), sdpa_kernel(SDPBackend.MATH):
    y=torch.nn.functional.scaled_dot_product_attention(q,k,v,is_causal=True,enable_gqa=True)
torch.cuda.synchronize()
assert torch.isfinite(y).all()
print('Chunked attention PASS; peak:',round(torch.cuda.max_memory_allocated()/2**30,2),'GiB')
"""
stress_code = "from pathlib import Path\nREPO_PATH=Path("+repr(str(REPO))+")\n" + stress_code
for gpu in ('0','1'):
    subprocess.run([sys.executable,'-u','-c',stress_code],check=True,
        env=dict(os.environ,CUDA_VISIBLE_DEVICES=gpu))


In [ ]:
import copy
import yaml
from huggingface_hub import HfApi, snapshot_download

cfg = yaml.safe_load((REPO/'configs/task3/kaggle_v3.yaml').read_text(encoding='utf-8'))
MODEL = cfg['model']['name_or_path']
OUT = WORK/cfg['experiment']['id']
OUT.mkdir(exist_ok=True)
# Tái sử dụng revision đã ghi khi resume, không tự lấy main mới giữa hai phiên.
revision_file = OUT/'model_revision.json'
if revision_file.exists():
    model_info = json.loads(revision_file.read_text(encoding='utf-8'))
    assert model_info['model'] == MODEL
    REVISION = model_info['revision']
else:
    REVISION = HfApi().model_info(MODEL).sha
    revision_file.write_text(json.dumps({'model':MODEL,'revision':REVISION}), encoding='utf-8')
snapshot_download(MODEL, revision=REVISION,
    allow_patterns=['*.json', '*.safetensors', '*.model', '*.txt', '*.jinja'])
cfg['model']['revision'] = REVISION
cfg['data'].update(train_path=str(TRAIN_JSON), input_path=str(TRAIN_JSON))
cfg['saving']['output_dir'] = str(OUT/'smoke')
cfg['model']['cache_path'] = str(OUT/'smoke/generation_cache.jsonl')
CONFIG = OUT/'smoke-config.json'
CONFIG.write_text(json.dumps(cfg,ensure_ascii=False,indent=2),encoding='utf-8')
print('Model:', MODEL, 'revision:', REVISION)


In [ ]:
# Kiểm tra source mới; không sửa source bằng replace chuỗi trong phiên chạy.
import inspect
from llm_grading.prompting.pipeline import ModelPipeline
from llm_grading.training.dataset import build_training_dataset
assert 'render_prompts' in inspect.signature(build_training_dataset).parameters
assert 'verify_diagnosis' in inspect.getsource(ModelPipeline)
assert cfg['task3']['fallback_on_failure'] is False
print('Source và cấu hình review1 đã sẵn sàng.')


In [ ]:
# Smoke trên GPU 0. Không nạp model vào kernel notebook.
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', TOKENIZERS_PARALLELISM='false',
           PYTHONUNBUFFERED='1')
subprocess.run([sys.executable, '-u', str(REPO/'scripts/predict.py'),
    '--config', str(CONFIG), '--input', str(PUBLIC_JSON), '--max-samples','3',
    '--output',str(OUT/'smoke_predictions.json')], env=env, check=True)
print((OUT/'smoke_predictions.json').read_text(encoding='utf-8'))


In [ ]:
# Kiểm tra 8 mẫu có nhãn, trải đều level; thay bằng len(train) để kiểm tra tất cả.
# Đây là kiểm tra trước khi chạy, không phải validation đủ lớn để ước lượng leaderboard.
from collections import defaultdict
from sklearn.metrics import precision_recall_fscore_support
from sklearn.preprocessing import MultiLabelBinarizer
from llm_grading.data.taxonomy import ERROR_LABELS

groups = defaultdict(list)
for row in train:
    groups[row['feedback_level']].append(row)
validation_rows = []
for offset in range(max(map(len, groups.values()))):
    for level in sorted(groups):
        if offset < len(groups[level]) and len(validation_rows) < 8:
            validation_rows.append(groups[level][offset])
VALIDATION_INPUT = OUT/'validation-input.json'
# Flat normalized input; reference feedback/labels are removed from the query.
validation_queries = []
for row in validation_rows:
    query = {k:v for k,v in row.items()
             if k not in {'feedback','error_labels','rubric','total_score','_has_reference'}}
    query['error_labels_missing'] = True
    validation_queries.append(query)
VALIDATION_INPUT.write_text(json.dumps(validation_queries,ensure_ascii=False,indent=2),encoding='utf-8')
val_cfg = copy.deepcopy(cfg)
val_cfg['saving']['output_dir'] = str(OUT/'validation')
val_cfg['model']['cache_path'] = str(OUT/'validation/generation_cache.jsonl')
val_cfg['retrieval']['exclude_same_exam'] = True
val_path = OUT/'validation-config.json'
val_path.write_text(json.dumps(val_cfg,ensure_ascii=False,indent=2),encoding='utf-8')
subprocess.run([sys.executable,'-u',str(REPO/'scripts/predict.py'),
    '--config',str(val_path),'--input',str(VALIDATION_INPUT),'--resume',
    '--output',str(OUT/'validation_predictions.json')],env=env,check=True)
validation_pred = json.loads((OUT/'validation_predictions.json').read_text(encoding='utf-8'))
assert [r['sample_id'] for r in validation_pred] == [r['sample_id'] for r in validation_rows]
mlb = MultiLabelBinarizer(classes=list(ERROR_LABELS))
expected = mlb.fit_transform([r['error_labels'] for r in validation_rows])
actual = mlb.transform([r['output']['error_labels'] for r in validation_pred])
precision, recall, f1, _ = precision_recall_fscore_support(expected,actual,average='micro',zero_division=0)
report = {'count':len(validation_rows), 'label_micro_precision':precision,
    'label_micro_recall':recall, 'label_micro_f1':f1,
    'label_exact_match':float((expected==actual).all(axis=1).mean()),
    'note':'Small same-exam-excluded diagnostic set; not semantic feedback accuracy or leaderboard score.'}
(OUT/'validation_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print(report)
for ref, got in zip(validation_rows,validation_pred):
    print('\n',ref['sample_id'],'Level',ref['feedback_level'])
    print('Nhãn tham chiếu:',ref['error_labels'],'| Dự đoán:',got['output']['error_labels'])
    print('Model:',got['output']['feedback'])
    print('Giảng viên:',ref['feedback'])


## Chạy cả hai GPU

Sau khi đọc kết quả kiểm tra, chạy cell tiếp theo. Hai worker chia shard, có log/cache/checkpoint riêng. Lưu thư mục có hậu tố review1; resume chỉ nhận cấu hình/dữ liệu/prompt khớp. Nếu đổi cấu hình, đặt experiment.id mới. Không lấy output cũ làm đầu vào phục hồi.


In [ ]:
import copy
import selectors

workers = []
selector = selectors.DefaultSelector()
try:
    for gpu in range(2):
        shard_dir = OUT/f'shard-{gpu+1}'
        shard_dir.mkdir(exist_ok=True)
        shard_cfg = copy.deepcopy(cfg)
        shard_cfg['saving']['output_dir'] = str(shard_dir)
        shard_cfg['model']['cache_path'] = str(shard_dir/'generation_cache.jsonl')
        config_path = shard_dir/'config.json'
        config_path.write_text(json.dumps(shard_cfg,ensure_ascii=False,indent=2),encoding='utf-8')
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu),
                   TOKENIZERS_PARALLELISM='false', PYTHONUNBUFFERED='1')
        cmd = [sys.executable,'-u',str(REPO/'scripts/predict.py'),
            '--config',str(config_path),'--input',str(PUBLIC_JSON),
            '--shard',f'{gpu+1}/2','--resume','--output',str(shard_dir/'predictions.json')]
        p = subprocess.Popen(cmd,env=env,stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1)
        workers.append(p)
        selector.register(p.stdout, selectors.EVENT_READ, gpu)
    while selector.get_map():
        for key, _ in selector.select(timeout=1):
            line = key.fileobj.readline()
            if line:
                print(f'[GPU {key.data}] {line}',end='',flush=True)
            else:
                selector.unregister(key.fileobj)
                key.fileobj.close()
    codes = [p.wait() for p in workers]
    assert codes == [0,0], f'Worker lỗi: {codes}; xem log phía trên, chưa đóng gói kết quả.'
finally:
    selector.close()
    for p in workers:
        if p.poll() is None:
            p.terminate()
    for p in workers:
        p.wait()


In [ ]:
run('merge_predictions.py','--task','task3',
    '--shards',OUT/'shard-1/predictions.json',OUT/'shard-2/predictions.json',
    '--input',PUBLIC_JSON,'--output',OUT/'predictions.json')
pred=json.loads((OUT/'predictions.json').read_text(encoding='utf-8'))
assert {r['sample_id'] for r in pred} == set(levels)
assert all(r['output']['feedback_level']==levels[r['sample_id']] for r in pred)
run('validate_predictions.py','--task','task3','--input',OUT/'predictions.json')
assert not any(r['output'].get('fallback') for r in pred), 'Có fallback: xem lại trước khi nộp'
assert all(r['output']['compliance']['pass'] for r in pred)
run('package_predictions.py','--task','task3','--input',OUT/'predictions.json',
    '--output',WORK/'submission_task3_qwen3_4b_review1.json')
print('Tải /kaggle/working/submission_task3_qwen3_4b_review1.json. Nộp file JSON này; giữ logs/cache riêng tư.')


In [ ]:
# Báo cáo chất lượng vận hành; fallback không đồng nghĩa chẩn đoán tốt.
from collections import Counter
fallbacks = [r for r in pred if r['output'].get('fallback')]
print('Tổng bài:', len(pred), '| Fallback:', len(fallbacks))
print('Fallback theo level:', dict(Counter(r['output']['feedback_level'] for r in fallbacks)))
(OUT/'fallback_report.json').write_text(json.dumps({
    'count':len(fallbacks), 'sample_ids':[r['sample_id'] for r in fallbacks]
},ensure_ascii=False,indent=2),encoding='utf-8')
print('Xem nguyên nhân trong shard-1/responses.jsonl và shard-2/responses.jsonl')
